# 10 - India vs US comparison

Objective: compare selected Indian and US stock returns using supplementary adjusted closes and prior-observation USD/INR, then inspect exploratory annual macro correlations.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [2]:
import pandas as pd

raw = ROOT / "data" / "raw"
india = sorted((raw / "india" / "stocks").glob("nse_*.parquet"))
usa = sorted((raw / "usa" / "stocks").glob("*.parquet")) if (raw / "usa" / "stocks").exists() else []
macro_dir = raw / "macroeconomic"
macro = [path for path in macro_dir.iterdir() if path.is_file() and path.name != ".gitkeep" and not path.name.lower().startswith("usdinr")] if macro_dir.exists() else []
fx = list(raw.rglob("*usd*inr*")) + list(raw.rglob("*inr*usd*"))
readiness = pd.DataFrame([
    {"input": "Annual India stock files", "available": bool(india), "count": len(india)},
    {"input": "US stock files", "available": bool(usa), "count": len(usa)},
    {"input": "USD/INR exchange-rate files", "available": bool(fx), "count": len(fx)},
    {"input": "World Bank macro series", "available": bool(macro), "count": len(macro)},
])
display(readiness)
comparison_dir = raw / "india" / "yahoo_adjusted"
fx_path = raw / "macroeconomic" / "usdinr_yahoo_2016_2025.parquet"
macro_path = raw / "macroeconomic" / "world_bank_annual_2016_2025.parquet"
if usa and fx_path.exists() and comparison_dir.exists():
    from finsight.comparison import load_adjusted_comparison_data, build_comparison_tables, build_macro_correlations
    prices = load_adjusted_comparison_data(comparison_dir, raw / "usa" / "stocks", fx_path)
    stock_metrics, market_summary, common_date_series = build_comparison_tables(prices)
    display(market_summary)
    display(stock_metrics.sort_values(["market", "currency_basis", "symbol"]))
    if macro_path.exists():
        macro_data = pd.read_parquet(macro_path)
        annual_alignment, macro_correlations = build_macro_correlations(common_date_series, macro_data)
        display(macro_correlations)
else:
    print("Comparison not calculated: supplementary adjusted closes and prior-date FX are required.")

,input,available,count
0,Annual India stock files,True,10
1,US stock files,True,10
2,USD/INR exchange-rate files,True,1
3,World Bank macro series,True,1


,market,currency_basis,stocks,median_total_return,median_annualized_return,median_annualized_volatility,median_max_drawdown,median_sharpe_zero_rf
0,INDIA,USD,9,1.888469,0.112016,0.266551,-0.520470,0.540262
1,INDIA,local_currency,9,2.829445,0.143725,0.263116,-0.483073,0.648448
2,USA,local_currency,10,5.920426,0.213529,0.284561,-0.462866,0.847688


,market,symbol,currency_basis,source_currency,observations,first_date,last_date,total_return,annualized_return,annualized_volatility,max_drawdown,sharpe_zero_rf
1,INDIA,HDFCBANK,USD,INR,2468,2016-01-04,2025-12-31,1.965542,0.114951,0.231520,-0.444842,0.595465
3,INDIA,ICICIBANK,USD,INR,2468,2016-01-04,2025-12-31,3.557695,0.163959,0.307362,-0.520470,0.657301
5,INDIA,INFY,USD,INR,2468,2016-01-04,2025-12-31,1.888469,0.112016,0.266551,-0.419425,0.540262
7,INDIA,ITC,USD,INR,2468,2016-01-04,2025-12-31,0.994387,0.071543,0.245986,-0.621064,0.410090
9,INDIA,ONGC,USD,INR,2468,2016-01-04,2025-12-31,0.860890,0.064138,0.335178,-0.740766,0.357801
11,INDIA,RELIANCE,USD,INR,2468,2016-01-04,2025-12-31,4.361447,0.183037,0.276266,-0.479704,0.758503
13,INDIA,SBIN,USD,INR,2468,2016-01-04,2025-12-31,2.630436,0.137757,0.333890,-0.632516,0.559560
15,INDIA,TCS,USD,INR,2468,2016-01-04,2025-12-31,1.479804,0.095165,0.241116,-0.379652,0.505061
17,INDIA,WIPRO,USD,INR,2468,2016-01-04,2025-12-31,1.010884,0.072427,0.262673,-0.547660,0.402307
0,INDIA,HDFCBANK,local_currency,INR,2469,2016-01-01,2025-12-31,2.960137,0.147570,0.224714,-0.410550,0.737753


,market,country_code,indicator_code,observations,pearson_r,pearson_pvalue,spearman_rho,spearman_pvalue
0,INDIA,IND,FP.CPI.TOTL.ZG,10,-0.020995,0.954094,0.078788,0.828717
1,INDIA,IND,NY.GDP.MKTP.KD.ZG,10,-0.045574,0.900514,-0.200000,0.579584
2,USA,USA,FP.CPI.TOTL.ZG,9,-0.619066,0.075460,-0.316667,0.406397
3,USA,USA,NY.GDP.MKTP.KD.ZG,10,-0.142736,0.694049,-0.042424,0.907364


Interpretation: this is a selected-stock comparison, not an index comparison. The Indian prices come from a supplementary Yahoo history rather than the original NSE Parquets; prior-observation FX is used to avoid same-date FX close look-ahead. Tata Motors is excluded because its Yahoo symbol history is ambiguous.

Conclusion: the World Bank annual macro sample is short and one USA CPI observation is missing. Correlations are exploratory, low-power, and not causal; see the generated report for assumptions and metrics.